# 202 Variable

View more, visit my tutorial page: https://morvanzhou.github.io/tutorials/
My Youtube Channel: https://www.youtube.com/user/MorvanZhou

Dependencies:
* torch: 1.0.0

Variable in torch is to build a computational graph,
but this graph is dynamic compared with a static graph in Tensorflow or Theano.
So torch does not have placeholder, torch can just pass variable to the computational graph.

https://ptorch.com/docs/4/pytorch-video-variable

神经网络中的参数都是 Variable 变量的形式。Pytorch 0.4以后，Variable已经和Tensor进行了整合，可以直接用tensor，tensor的用法和variable相同。  

要想使得Tensor具有autograd功能，只需要设置`tensor.requries_grad=True`。

---

~~`autograd.Variable`是Autograd中的核心类，它简单封装了Tensor，并支持几乎所有Tensor的操作。Tensor在被封装为Variable之后，可以调用它的`.backward`自动计算所有的梯度，来实现反向传播，自动计算所有梯度~~ ~~Variable的数据结构如图所示。~~

~~Variable主要包含三个属性。~~
~~- `data`：保存Variable变量中的原始张量Tensor~~
~~- `grad`：保存`data`对应的梯度，`grad`也是个Variable，而不是Tensor，它和`data`的形状一样。~~
~~- `grad_fn`：指向一个`Function`对象，这个`Function`用来反向传播计算输入的梯度，具体细节会在下一章讲解。~~

![图2-6:Variable的数据结构](images/autograd_Variable.png)

  *从0.4起, Variable 正式合并入Tensor, Variable 本来实现的自动微分功能，Tensor就能支持。还是可以使用Variable(tensor), 但是这个操作其实什么都没做。建议以后直接使用 tensor*. 

In [30]:
import torch

# tensor = torch.tensor([[1,2],[3,4]]) # int
tensor = torch.Tensor([[1,2],[3,4]]) # float

# only Tensors of floating point dtype can require gradients
tensor.requires_grad = True
tensor

tensor([[1., 2.],
        [3., 4.]], requires_grad=True)

计算梯度值用：`v_out.backward()`  
查看梯度值用：`variable.grad`

In [33]:
s_out = torch.mean(tensor*tensor)
print(s_out)
s_out.backward()
print(tensor.grad)

tensor(7.5000, grad_fn=<MeanBackward1>)
tensor([[1.5000, 3.0000],
        [4.5000, 6.0000]])


In [36]:
print(tensor) # torch.Tensor
print(tensor.detach()) # torch.Tensor
tensor.detach().numpy() # numpy.ndarray

tensor([[1., 2.],
        [3., 4.]], requires_grad=True)
tensor([[1., 2.],
        [3., 4.]])


array([[1., 2.],
       [3., 4.]], dtype=float32)

### 如下为Pytorch 0.4版本

In [23]:
import torch
from torch.autograd import Variable

In [24]:
# torch.Tensor([[1,2],[3,4]]) # int
tensor = torch.FloatTensor([[1,2],[3,4]])
variable = Variable(tensor, requires_grad=True)

print(tensor)
print(variable)

tensor([[1., 2.],
        [3., 4.]])
tensor([[1., 2.],
        [3., 4.]], requires_grad=True)


Till now the tensor and variable seem the same.

However, the variable is a part of the graph, it's a part of the auto-gradient.


In [3]:
t_out = torch.mean(tensor*tensor)       # x^2
v_out = torch.mean(variable*variable)   # x^2
print(t_out)
print(v_out)

tensor(7.5000)
tensor(7.5000, grad_fn=<MeanBackward1>)


In [4]:
v_out.backward()    # backpropagation from v_out

$$ v_{out} = {{1} \over {4}} sum(variable^2) $$

the gradients w.r.t the variable, 

$$ {d(v_{out}) \over d(variable)} = {{1} \over {4}} 2 variable = {variable \over 2}$$

得到每个variable对应的梯度值，let's check the result pytorch calculated for us below:

In [5]:
variable.grad

tensor([[0.5000, 1.0000],
        [1.5000, 2.0000]])

In [6]:
print(variable) # torch.Tensor
print(variable.detach()) # torch.Tensor
variable.detach().numpy() # numpy.ndarray

tensor([[1., 2.],
        [3., 4.]], requires_grad=True)
tensor([[1., 2.],
        [3., 4.]])


array([[1., 2.],
       [3., 4.]], dtype=float32)

Note that we did `.backward()` on `v_out` but `variable` has been assigned new values on it's `grad`.

As this line 
```python
v_out = torch.mean(variable*variable)
``` 
will make a new variable `v_out` and connect it with `variable` in computation graph.

In [7]:
type(v_out)

torch.Tensor

In [8]:
type(v_out.data)

torch.Tensor